In [1]:
!pip install faker sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 68.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime

fake = Faker()

In [3]:
customers = []

for i in range(1,1001):

    customers.append({
        "customer_id":i,
        "name":fake.name(),
        "email":fake.email(),
        "signup_date":fake.date_between(
            start_date="-3y",
            end_date="today"
        ),
        "city":fake.city()
    })

customers = pd.DataFrame(customers)

customers.head()

,customer_id,name,email,signup_date,city
0,1,Michael Andrade,ycantrell@example.org,2025-09-01,Stevenshire
1,2,Ann Myers,vazquezmelissa@example.net,2025-06-06,North Jenniferside
2,3,Jennifer Cruz,campbellanthony@example.com,2025-05-08,Zacharyside
3,4,Sandra Wilkerson,maloneeric@example.net,2024-09-17,Dillonville
4,5,Jesse Middleton,qcooper@example.net,2025-08-09,South Candace


In [4]:
customers = pd.concat([
    customers,
    customers.iloc[:20]
])

customers.shape

(1020, 5)

In [5]:
customers.loc[
    customers.sample(50).index,
    "email"
] = None

In [6]:
customers.loc[
    customers.sample(10).index,
    "signup_date"
] = "2035-01-01"

In [7]:
categories = [
    "Electronics",
    "Fashion",
    "Sports",
    "Home"
]

products = []

for i in range(1,201):

    price = round(
        random.uniform(10,5000),
        2
    )

    products.append({
        "product_id":i,
        "product_name":f"Product_{i}",
        "category":random.choice(categories),
        "price":price,
        "cost":round(price*0.6,2)
    })

products = pd.DataFrame(products)

In [8]:
products.loc[
    products.sample(5).index,
    "price"
] = -100

In [9]:
orders = []

for i in range(1,5001):

    orders.append({
        "order_id":i,
        "customer_id":random.randint(1,1000),
        "order_date":fake.date_between(
            start_date="-2y",
            end_date="today"
        ),
        "status":random.choice([
            "Completed",
            "Cancelled",
            "Returned",
            "Pending"
        ])
    })

orders = pd.DataFrame(orders)

orders.head()

,order_id,customer_id,order_date,status
0,1,910,2026-04-19,Returned
1,2,933,2024-09-19,Cancelled
2,3,81,2025-06-15,Returned
3,4,608,2024-08-23,Completed
4,5,668,2025-05-26,Completed


In [10]:
order_items = []

for i in range(1,15001):

    product = random.randint(1,200)

    price = products.loc[
        products["product_id"] == product,
        "price"
    ].values[0]

    order_items.append({
        "order_item_id":i,
        "order_id":random.randint(1,5000),
        "product_id":product,
        "quantity":random.randint(1,5),
        "unit_price":price
    })

order_items = pd.DataFrame(order_items)

order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price
0,1,4137,8,4,652.30
1,2,4363,10,2,974.25
2,3,921,77,3,2559.59
3,4,1681,105,1,4959.45
4,5,2128,148,4,3250.66


In [11]:
customers.to_csv(
    "customers_raw.csv",
    index=False
)

products.to_csv(
    "products_raw.csv",
    index=False
)

orders.to_csv(
    "orders_raw.csv",
    index=False
)

order_items.to_csv(
    "order_items_raw.csv",
    index=False
)

In [12]:
from google.colab import files

files.download("customers_raw.csv")
files.download("products_raw.csv")
files.download("orders_raw.csv")
files.download("order_items_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
customers_clean = customers.copy()

customers_clean = customers_clean.drop_duplicates(
    subset="customer_id"
)

customers_clean = customers_clean.dropna(
    subset=["email"]
)

customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"]
)

customers_clean = customers_clean[
    customers_clean["signup_date"]
    <= pd.Timestamp.today()
]

In [14]:
products_clean = products.copy()

products_clean = products_clean[
    products_clean["price"] > 0
]

In [15]:
report = {
    "duplicates_removed":
    len(customers)-len(customers.drop_duplicates()),

    "missing_emails":
    customers["email"].isna().sum(),

    "negative_prices":
    (products["price"] < 0).sum()
}

report

{'duplicates_removed': 20,
 'missing_emails': np.int64(52),
 'negative_prices': np.int64(5)}

In [16]:
merged = (
    orders
    .merge(order_items,on="order_id")
)

merged["revenue"] = (
    merged["quantity"]
    *
    merged["unit_price"]
)

In [17]:
merged["order_date"] = pd.to_datetime(
    merged["order_date"]
)

monthly_revenue = (
    merged
    .groupby(
        pd.Grouper(
            key="order_date",
            freq="M"
        )
    )["revenue"]
    .sum()
)

monthly_revenue

/tmp/ipykernel_1663/3190869183.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  pd.Grouper(


,revenue
order_date,
2024-06-30,1328665.61
2024-07-31,3729082.07
2024-08-31,4086270.75
2024-09-30,4251651.10
2024-10-31,4627728.76
2024-11-30,4516070.58
2024-12-31,4334369.90
2025-01-31,3878787.55
2025-02-28,3903921.01


In [18]:
customer_segments = (
    merged
    .groupby("customer_id")["revenue"]
    .sum()
    .reset_index()
)

customer_segments["segment"] = np.where(
    customer_segments["revenue"] > 50000,
    "VIP",
    np.where(
        customer_segments["revenue"] > 20000,
        "Gold",
        "Regular"
    )
)

customer_segments.head()

,customer_id,revenue,segment
0,1,68855.77,VIP
1,2,144858.63,VIP
2,3,85437.81,VIP
3,4,42977.61,Gold
4,5,82146.57,VIP


In [19]:
customers_clean.to_csv(
    "customers_clean.csv",
    index=False
)

products_clean.to_csv(
    "products_clean.csv",
    index=False
)

orders.to_csv(
    "orders_clean.csv",
    index=False
)

order_items.to_csv(
    "order_items_clean.csv",
    index=False
)

In [20]:
from google.colab import files

files.download("customers_clean.csv")
files.download("products_clean.csv")
files.download("orders_clean.csv")
files.download("order_items_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>